In [13]:
!gdown 1zxqvzv9Ii0wR7Jmu4cM_w7hoMdlAPnqZ -O aoi.geojson

Downloading...
From: https://drive.google.com/uc?id=1zxqvzv9Ii0wR7Jmu4cM_w7hoMdlAPnqZ
To: /content/aoi.geojson
100% 481/481 [00:00<00:00, 2.98MB/s]


In [14]:
# ──────────────────────────────────────────────────────────────
# 0. Instalación de dependencias  (se ejecuta una sola vez)
# ──────────────────────────────────────────────────────────────
!pip -q install pystac-client stackstac rasterio rioxarray geopandas

# ──────────────────────────────────────────────────────────────
# 1. Parámetros de entrada
# ──────────────────────────────────────────────────────────────
import datetime as dt
from pathlib import Path

AOI_FILE   = "aoi.geojson"                 # ruta al AOI
FECHAS     = ["2025-02-19", "2025-03-11"]  # dos fechas puntuales (AAAA-MM-DD)
MAX_NUBES  = 20                          # % máximo de nubes
OUT_DIR    = Path("sentinel_descargas")    # carpeta de salida
BANDAS_10  = ["B02", "B03", "B04", "B08"]  # 10 m: azul, verde, rojo, NIR
BANDAS_20  = ["B8A", "B11", "B12"]         # 20 m: NIR estrecho, SWIR1, SWIR2
TODAS_BANDAS = BANDAS_10 + BANDAS_20

OUT_DIR.mkdir(exist_ok=True)

# ──────────────────────────────────────────────────────────────
# 2. Cargar AOI y calcular bounding box
# ──────────────────────────────────────────────────────────────
import geopandas as gpd
aoi_gdf = gpd.read_file(AOI_FILE).to_crs(4326)
bbox = aoi_gdf.total_bounds  # [minx, miny, maxx, maxy]

# ──────────────────────────────────────────────────────────────
# 3. Funciones auxiliares
# ──────────────────────────────────────────────────────────────
from pystac_client import Client
import stackstac
import rioxarray as rxr
import rasterio
from rasterio.merge import merge

In [15]:
# Conecto al catálogo STAC de Earth-Search (Sentinel-2 L2A COGs)
STAC_URL = "https://earth-search.aws.element84.com/v1"
catalogo = Client.open(STAC_URL)

# -----------------------------------------
#  Conexión: Earth-Search (L1C)
# -----------------------------------------
STAC_URL = "https://earth-search.aws.element84.com/v1"
catalogo = Client.open(STAC_URL)

def buscar_items(fecha_iso, dias_max=5, nubosidad_max=100):
    fecha = dt.date.fromisoformat(fecha_iso)

    for delta in range(dias_max + 1):
        rango = f"{fecha - dt.timedelta(days=delta)}/{fecha + dt.timedelta(days=delta)}"
        res = catalogo.search(
            collections=["sentinel-2-l1c"],      # <-- L1C
            bbox=bbox,
            datetime=rango,
            query={"eo:cloud_cover": {"lt": nubosidad_max}}
        )
        items = list(res.items())               # .items() ≈ get_items()
        if items:
            print(f"   ➜ {len(items)} item(s) L1C encontrados ±{delta} d")
            items.sort(key=lambda i: i.properties["eo:cloud_cover"])
            return items

    raise ValueError(f"Sin L1C para {fecha_iso} en ±{dias_max} d")

# ─── Mapeo "código banda"  ➜  "asset key" ───
BAND_KEY = {
    "B02": "blue",
    "B03": "green",
    "B04": "red",
    "B08": "nir",
    "B8A": "nir08",
    "B11": "swir16",
    "B12": "swir22",
}

TODAS_BANDAS = list(BAND_KEY.keys())      # mantiene tu lista original

def descargar_y_mosaicar(items, fecha_iso):
    """Descarga las bandas indicadas, hace mosaico y crea un GeoTIFF multibanda."""
    archivos_temp = {b: [] for b in TODAS_BANDAS}

    for item in items:
        for banda in TODAS_BANDAS:
          asset_key = BAND_KEY[banda]                 # <── nuevo
          url = item.assets[asset_key].href           # antes era item.assets[banda]
          # acceso HTTPS en lugar de s3://
          url = url.replace(
              "s3://sentinel-s2-l1c",
              "https://sentinel-s2-l1c.s3.amazonaws.com"
          )
          nombre = OUT_DIR / f"{item.id}_{banda}.tif"
          if not nombre.exists():
              rxr.open_rasterio(url, masked=True).rio.to_raster(nombre)
          archivos_temp[banda].append(nombre)

    # Mosaico banda a banda ─ se guarda en memoria y luego se escribe un multi-banda
    mosaicos = []
    meta_ref = None

    for banda in TODAS_BANDAS:
        srcs = [rasterio.open(p) for p in archivos_temp[banda]]
        mosaic, out_trans = merge(srcs)
        if meta_ref is None:
            meta_ref = srcs[0].meta.copy()
            meta_ref.update({
                "driver": "GTiff",
                "height": mosaic.shape[1],
                "width": mosaic.shape[2],
                "count": len(TODAS_BANDAS),
                "transform": out_trans,
                "dtype": mosaic.dtype
            })
        mosaicos.append(mosaic[0])
        for s in srcs: s.close()

    # Stack: R,G,B,8A,12,8,11  (orden pensado para visualización + análisis de agua)
    orden = ["B04","B03","B02","B8A","B12","B08","B11"]
    stack = [mosaicos[TODAS_BANDAS.index(b)] for b in orden]

    # Guardar archivo final
    salida = OUT_DIR / f"AOI_{fecha_iso.replace('-','')}.tif"
    with rasterio.open(salida, "w", **meta_ref) as dst:
        for i, banda_arr in enumerate(stack, start=1):
            dst.write(banda_arr, i)
            dst.set_band_description(i, orden[i-1])

    print(f"✔️  Imagen guardada: {salida}")

# ──────────────────────────────────────────────────────────────
# 4. Bucle principal
# ──────────────────────────────────────────────────────────────
for fecha in FECHAS:
    print(f"\n🛰️  Procesando {fecha}…")
    items = buscar_items(fecha)
    descargar_y_mosaicar(items, fecha)


🛰️  Procesando 2025-02-19…
   ➜ 2 item(s) L1C encontrados ±0 d


/usr/local/lib/python3.12/dist-packages/IPython/core/async_helpers.py:78: SerializationWarning: saving variable None with floating point data as an integer dtype without any _FillValue to use for NaNs
  coro.send(None)
/usr/local/lib/python3.12/dist-packages/IPython/core/async_helpers.py:78: SerializationWarning: saving variable None with floating point data as an integer dtype without any _FillValue to use for NaNs
  coro.send(None)
/usr/local/lib/python3.12/dist-packages/IPython/core/async_helpers.py:78: SerializationWarning: saving variable None with floating point data as an integer dtype without any _FillValue to use for NaNs
  coro.send(None)
/usr/local/lib/python3.12/dist-packages/IPython/core/async_helpers.py:78: SerializationWarning: saving variable None with floating point data as an integer dtype without any _FillValue to use for NaNs
  coro.send(None)
/usr/local/lib/python3.12/dist-packages/IPython/core/async_helpers.py:78: SerializationWarning: saving variable None with fl

✔️  Imagen guardada: sentinel_descargas/AOI_20250219.tif

🛰️  Procesando 2025-03-11…
   ➜ 2 item(s) L1C encontrados ±0 d


/usr/local/lib/python3.12/dist-packages/IPython/core/async_helpers.py:78: SerializationWarning: saving variable None with floating point data as an integer dtype without any _FillValue to use for NaNs
  coro.send(None)
/usr/local/lib/python3.12/dist-packages/IPython/core/async_helpers.py:78: SerializationWarning: saving variable None with floating point data as an integer dtype without any _FillValue to use for NaNs
  coro.send(None)
/usr/local/lib/python3.12/dist-packages/IPython/core/async_helpers.py:78: SerializationWarning: saving variable None with floating point data as an integer dtype without any _FillValue to use for NaNs
  coro.send(None)
/usr/local/lib/python3.12/dist-packages/IPython/core/async_helpers.py:78: SerializationWarning: saving variable None with floating point data as an integer dtype without any _FillValue to use for NaNs
  coro.send(None)
/usr/local/lib/python3.12/dist-packages/IPython/core/async_helpers.py:78: SerializationWarning: saving variable None with fl

✔️  Imagen guardada: sentinel_descargas/AOI_20250311.tif


In [16]:
!pip -q install rasterio geopandas

from pathlib import Path
import numpy as np
import geopandas as gpd
import rasterio
from rasterio.mask import mask
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.enums import Compression
from rasterio.transform import array_bounds

# --- ARCHIVO AOI y carpeta de TIFF de 10 m ---
AOI_FILE = "aoi.geojson"
CARPETA  = Path("sentinel_descargas")      # donde están AOI_2025*.tif

# --- leer polígono AOI una sola vez ---
aoi_gdf  = gpd.read_file(AOI_FILE)
aoi_geom = aoi_gdf.unary_union            # shapely geometry

def procesar_tiff(path_tif, nueva_res=20):
    """
    Recorta el GeoTIFF al AOI, lo re-muestra a 'nueva_res' (m)
    y lo guarda comprimido (DEFLATE) con sufijo _20m_clip.tif
    """
    out_path = path_tif.with_name(path_tif.stem + "_20m_clip.tif")
    if out_path.exists():
        print(f"⏩ {out_path.name} ya existe, salto.")
        return

    with rasterio.open(path_tif) as src:
        # 1) reproyectar AOI al CRS del raster y recortar
        geom_proj = [g.to_crs(src.crs).geometry[0] if isinstance(g, gpd.GeoDataFrame)
                     else g for g in [aoi_gdf.to_crs(src.crs)]]

        datos_clip, trans_clip = mask(src, geom_proj, crop=True)
        datos_clip = np.ma.filled(datos_clip, src.nodata
                                  if src.nodata is not None else 0)

        # 2) calcular nueva malla a 20 m
        h_clip, w_clip = datos_clip.shape[1:]
        left, bottom, right, top = array_bounds(h_clip, w_clip, trans_clip)

        dst_tr, dst_w, dst_h = calculate_default_transform(
            src.crs, src.crs, w_clip, h_clip,
            left, bottom, right, top,
            resolution=nueva_res
        )

        # 3) meta destino con compresión
        dst_meta = src.meta.copy()
        dst_meta.update({
            "height":   dst_h,
            "width":    dst_w,
            "transform": dst_tr,
            "compress":  "DEFLATE",
            "predictor": 2,
            "tiled":     True,
            "blockxsize": 512,
            "blockysize": 512
        })

        # 4) crear archivo destino y re-muestra banda por banda
        with rasterio.open(out_path, "w", **dst_meta) as dst:
            for i in range(src.count):
                dst_arr = np.empty((dst_h, dst_w), dtype=datos_clip.dtype)

                reproject(
                    source       = datos_clip[i],
                    destination  = dst_arr,
                    src_transform= trans_clip,
                    src_crs      = src.crs,
                    dst_transform= dst_tr,
                    dst_crs      = src.crs,
                    resampling   = Resampling.average
                )
                dst.write(dst_arr, i + 1)
                dst.set_band_description(i + 1, src.descriptions[i])

    print(f"✔️  {out_path.name} generado (recorte + 20 m + compresión)")

# --- Procesar todo lo que empiece con AOI_*.tif ---
for tif in CARPETA.glob("AOI_*.tif"):
    procesar_tiff(tif)

/tmp/ipykernel_401/1014891869.py:18: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  aoi_geom = aoi_gdf.unary_union            # shapely geometry


✔️  AOI_20250219_20m_clip.tif generado (recorte + 20 m + compresión)
✔️  AOI_20250311_20m_clip.tif generado (recorte + 20 m + compresión)


### Exporting `sentinel_descargas` to Google Drive

First, we need to mount your Google Drive to access it from this Colab environment. Follow the prompts to authenticate and grant access.

In [17]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Now that your Google Drive is mounted, you can copy the `sentinel_descargas` folder to your Drive. It will be copied to the root of your 'My Drive' by default. If you want to place it in a specific folder, replace `MyDrive` with the path to that folder (e.g., `MyDrive/my_data_folder`).

In [19]:
import shutil
import os

source_folder = 'sentinel_descargas'
destination_folder = '/content/drive/MyDrive/GIS/TP2/sentinel_descargas'

# Create the destination folder in Drive if it doesn't exist
os.makedirs(destination_folder, exist_ok=True)

# Copy the contents of the source folder to the destination folder
# use shutil.copytree for folders, or shutil.copy for files

if os.path.exists(source_folder):
    # Remove existing folder in destination to avoid shutil.copytree error if it exists
    if os.path.exists(destination_folder):
        shutil.rmtree(destination_folder)
    shutil.copytree(source_folder, destination_folder)
    print(f"Folder '{source_folder}' successfully copied to '{destination_folder}'")
else:
    print(f"Source folder '{source_folder}' does not exist.")

Folder 'sentinel_descargas' successfully copied to '/content/drive/MyDrive/GIS/TP2/sentinel_descargas'
